# 04 — Lighting: Run 2 vs Run 3

**The problem:** The production classifier degrades under suboptimal lighting (underexposure, overexposure, mixed color temperature). This is a training distribution mismatch — the model was trained on well-lit dermoscopy images and never saw lighting variation. 

**The fix:** Train Run 3 with aggressive lighting augmentation. Same architecture, same class weighting as Run 2. Only the training augmentation changes.

**The evidence:** We already have a brightness-binned benchmark from notebook 03 (dark / medium / light tertiles). Rerunning that on both models shows whether the augmentation reduced the performance gap across lighting conditions.

---

**What changed in the augmentation (Run 2 → Run 3):**

| Augmentation | Run 2 (standard) | Run 3 (lighting-robust) |
|---|---|---|
| ColorJitter brightness | 0.2 | 0.5 |
| ColorJitter contrast | 0.2 | 0.5 |
| ColorJitter saturation | 0.2 | 0.4 |
| ColorJitter hue | — | 0.1 |
| RandomAutocontrast | — | p=0.3 |
| RandomEqualize | — | p=0.2 |
| GaussianBlur | — | σ=0.1–1.5 |

`brightness=0.5` means the model sees images at 50–150% of original exposure during training — covers both underexposed bedside photos and overexposed fluorescent-lit nurseries.

## Setup

In [1]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image

from data import get_dataloaders
from model import build_model

DATA_DIR   = '../data'
RESULTS_DIR = Path('../results')
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: mps


## Train Run 3 — lighting-robust augmentation + class weighting

Identical to Run 2 except `lighting_robust=True` in the dataloader, which activates the stronger augmentation pipeline in `src/data.py`.

In [2]:
# lighting_robust=True only affects the train split — val/test get standard eval transforms
train_loader, val_loader, test_loader = get_dataloaders(DATA_DIR, batch_size=32, lighting_robust=True)

class_names = train_loader.dataset.classes
num_classes  = len(class_names)
print(f'Classes: {class_names}')
print(f'Train / Val / Test: {len(train_loader.dataset)} / {len(val_loader.dataset)} / {len(test_loader.dataset)}')

Classes: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
Train / Val / Test: 5229 / 1120 / 1121


In [3]:
# Class weights: same formula as Run 2 — inverse frequency, normalized to average 1
train_dir = Path(DATA_DIR) / 'train'
counts = np.array([
    len(list((train_dir / cls).glob('*.jpg')))
    for cls in class_names
], dtype=np.float32)

weights = 1.0 / counts
weights = weights / weights.sum() * num_classes
class_weights = torch.tensor(weights).to(DEVICE)

print('Class weights (same as Run 2):')
for cls, w in zip(class_names, weights):
    print(f'  {cls:<8} {w:.4f}')

Class weights (same as Run 2):
  akiec    0.8928
  bcc      0.6199
  bkl      0.2789
  df       2.7834
  mel      0.3301
  nv       0.0375
  vasc     2.0573


In [ ]:
model_v3 = build_model(num_classes=num_classes, backbone='resnet50', freeze_backbone=True)
model_v3 = model_v3.to(DEVICE)

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_v3.parameters()), lr=1e-3
)
criterion = nn.CrossEntropyLoss(weight=class_weights)

(RESULTS_DIR / 'v3').mkdir(parents=True, exist_ok=True)

history_v3 = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0

for epoch in range(5):
    model_v3.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model_v3(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    model_v3.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model_v3(images)
            val_loss += criterion(outputs, labels).item()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    train_loss = running_loss / len(train_loader)
    val_loss  /= len(val_loader)
    val_acc    = correct / total

    history_v3['train_loss'].append(train_loss)
    history_v3['val_loss'].append(val_loss)
    history_v3['val_acc'].append(val_acc)

    print(f'Epoch {epoch+1}/5 | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model_v3.state_dict(), RESULTS_DIR / 'v3' / 'best_model.pt')

print(f'\nBest val acc (Run 3): {best_val_acc:.4f}')

Epoch 1/5 | train_loss=1.7790 | val_loss=1.3522 | val_acc=0.4929


Val accuracy may come in slightly lower than Run 2 — that's expected. Lighting augmentation deliberately makes training harder (the model has to learn invariances it didn't before). The payoff shows up in the brightness-binned benchmark below, not in raw accuracy.

## Lighting robustness benchmark

We measure robustness using the same brightness-tertile proxy from notebook 03: bin the 1121 test images into dark / medium / light thirds by mean luminance, then compare accuracy per bin between Run 2 and Run 3.

A smaller gap across bins = better lighting robustness. We expect Run 3 to close the gap, particularly lifting the underperforming bin.

In [ ]:
def mean_luminance(path):
    img = Image.open(path).convert('RGB')
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return float(0.299 * arr[..., 0].mean() + 0.587 * arr[..., 1].mean() + 0.114 * arr[..., 2].mean())

samples    = test_loader.dataset.samples
gt_labels  = np.array([s[1] for s in samples])
brightness = np.array([mean_luminance(p) for p, _ in samples])

edges    = np.quantile(brightness, [1/3, 2/3])
tone_bin = np.where(brightness <= edges[0], 'dark',
            np.where(brightness <= edges[1], 'medium', 'light'))

print(f'Brightness — min: {brightness.min():.3f}, median: {np.median(brightness):.3f}, max: {brightness.max():.3f}')
print(f'Bin sizes: {dict(zip(*np.unique(tone_bin, return_counts=True)))}')

In [ ]:
def run_inference(model, loader, device):
    """Returns (preds, labels) numpy arrays over the full loader."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            all_preds.append(model(images).argmax(1).cpu().numpy())
            all_labels.append(labels.numpy())
    return np.concatenate(all_preds), np.concatenate(all_labels)


# Run 2 — load from checkpoint
model_v2 = build_model(num_classes=num_classes, backbone='resnet50', freeze_backbone=True)
model_v2.load_state_dict(torch.load(RESULTS_DIR / 'v2' / 'best_model.pt', map_location=DEVICE))
model_v2 = model_v2.to(DEVICE)
preds_v2, labels_v2 = run_inference(model_v2, test_loader, DEVICE)

# Run 3 — load from checkpoint (just trained above)
model_v3_eval = build_model(num_classes=num_classes, backbone='resnet50', freeze_backbone=True)
model_v3_eval.load_state_dict(torch.load(RESULTS_DIR / 'v3' / 'best_model.pt', map_location=DEVICE))
model_v3_eval = model_v3_eval.to(DEVICE)
preds_v3, labels_v3 = run_inference(model_v3_eval, test_loader, DEVICE)

print(f'Run 2 overall test acc: {(preds_v2 == labels_v2).mean():.4f}')
print(f'Run 3 overall test acc: {(preds_v3 == labels_v3).mean():.4f}')

In [ ]:
# Per-bin accuracy for both runs
df_v2 = pd.DataFrame({'tone_bin': tone_bin, 'correct': (preds_v2 == gt_labels).astype(int)})
df_v3 = pd.DataFrame({'tone_bin': tone_bin, 'correct': (preds_v3 == gt_labels).astype(int)})

summary = pd.DataFrame({
    'n':           df_v2.groupby('tone_bin')['correct'].size(),
    'Run 2 acc':   df_v2.groupby('tone_bin')['correct'].mean().round(3),
    'Run 3 acc':   df_v3.groupby('tone_bin')['correct'].mean().round(3),
})
summary['delta'] = (summary['Run 3 acc'] - summary['Run 2 acc']).round(3)

print(summary.to_string())
print(f"\nRun 2 gap (best−worst bin): {summary['Run 2 acc'].max() - summary['Run 2 acc'].min():.3f}")
print(f"Run 3 gap (best−worst bin): {summary['Run 3 acc'].max() - summary['Run 3 acc'].min():.3f}")

In [ ]:
bins   = ['dark', 'medium', 'light']
x      = np.arange(len(bins))
width  = 0.35

acc_v2 = [summary.loc[b, 'Run 2 acc'] for b in bins]
acc_v3 = [summary.loc[b, 'Run 3 acc'] for b in bins]

fig, ax = plt.subplots(figsize=(7, 5))
bars2 = ax.bar(x - width/2, acc_v2, width, label='Run 2 — standard aug',        color='steelblue', alpha=0.85)
bars3 = ax.bar(x + width/2, acc_v3, width, label='Run 3 — lighting-robust aug',  color='darkorange', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(bins)
ax.set_xlabel('Brightness bin (proxy for lighting condition)')
ax.set_ylabel('Test accuracy')
ax.set_ylim(0.55, 0.85)
ax.set_title('Lighting robustness: Run 2 vs Run 3\n(smaller gap across bins = more robust to lighting variation)')
ax.axhline(summary['Run 2 acc'].mean(), color='steelblue', linestyle=':', linewidth=1, alpha=0.6)
ax.axhline(summary['Run 3 acc'].mean(), color='darkorange', linestyle=':', linewidth=1, alpha=0.6)
ax.legend()

# annotate gap
gap_v2 = summary['Run 2 acc'].max() - summary['Run 2 acc'].min()
gap_v3 = summary['Run 3 acc'].max() - summary['Run 3 acc'].min()
ax.text(0.98, 0.05,
        f'Run 2 gap: {gap_v2:.3f}\nRun 3 gap: {gap_v3:.3f}',
        transform=ax.transAxes, ha='right', va='bottom',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'lighting_robustness_comparison.png', dpi=150)
plt.show()

## Interpretation

**What to look for:**
- Did the gap between best and worst bins shrink? That's the robustness win.
- Did the weakest bin (usually "light" in Run 2) improve? That's the fix working.
- Did overall accuracy stay comparable? A small drop is acceptable — we're trading raw accuracy for consistency across lighting conditions.

**Why the gap matters clinically:**  
In a neonatal setting, lighting is uncontrolled — bedside lamps, overhead fluorescents, window light, phone flashlights. A model that performs 7pp worse under one lighting condition than another will systematically fail for certain clinical settings. Run 3 addresses this at training time, with no change to inference infrastructure.

**Caveats:**  
The brightness proxy is still HAM10000 dermoscopy brightness, which mostly tracks lesion color rather than ambient lighting. The technique is right; measuring it on neonatal data under controlled lighting variation would be the real validation. Temperature scaling (see notebook 03 Part B) is still the right next step for calibration.

**Production roadmap:**
1. Aggressive augmentation at training time (this notebook — done)
2. CLAHE illumination normalization as a preprocessing step at inference — no retraining needed, cuts residual gap
3. Test-time augmentation (TTA) over lighting variants — averages out remaining sensitivity at inference cost
4. Real validation with neonatal images captured under varied lighting conditions